# Case Study 3: time-dependent curvilinear simulation

This notebook creates non-overlapping particles inside the peristaltic map at `t = 0`. It uses `maps.py` for the explicit time-dependent map and `pre_post_processing_v19.py` for area-weighted sampling and physical-space overlap rejection. The next step is to run the time-dependent curvilinear solver from the same live variables.


In [ ]:
import importlib
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.collections import LineCollection

import maps
import pre_post_processing_v19 as pp
import time_varying_curvilinear_solver as csol

maps = importlib.reload(maps)
pp = importlib.reload(pp)
csol = importlib.reload(csol)

FIGURE_DIR = Path("Figures")
VIDEO_DIR = Path("Videos")
FIGURE_DIR.mkdir(exist_ok=True)
VIDEO_DIR.mkdir(exist_ok=True)
plt.rcParams["animation.embed_limit"] = 500


## Parameters


In [ ]:
t0 = 0.0
N = 1000
particle_radius = 0.02
particle_mass = 1.0
velocity = (0.0, 0.0)

fraction = 1.0
r_min = 0.05
r_max = 0.90
safety = 1.08
max_trials = 500_000
seed = 3

print('map constants:')
print('a     =', float(maps.A))
print('alpha =', float(maps.ALPHA))
print('b     =', float(maps.B0))
print('eps   =', float(maps.EPS))
print('M     =', float(maps.M))
print('omega =', float(maps.OMEGA))


## Create Initial Particles


In [ ]:
q0, p0, x0, m, rad, group_mobile, nearest, info = pp.make_initial_conditions(
    N=N,
    particle_radius=particle_radius,
    particle_mass=particle_mass,
    velocity=velocity,
    t=t0,
    fraction=fraction,
    r_min=r_min,
    r_max=r_max,
    safety=safety,
    max_trials=max_trials,
    seed=seed,
)

info


## Visual Check


In [ ]:
fig, axes = pp.plot_initial_configuration(q0, x0, t=t0, s=3)
fig.savefig(FIGURE_DIR / "case_3_initial_particles_t0.png", dpi=300, bbox_inches="tight")
plt.show()


## Next: Time-Dependent Solver

The notebook has `q0`, `p0`, `x0`, `m`, `rad`, and `group_mobile` in memory. The solver module is imported as `csol` from `time_varying_curvilinear_solver.py`, so the next cells can follow the Case Study 1 pattern without saving particles to disk.


## Simulation Parameters

The peristaltic shape repeats after one period `2? / omega`. The run below covers four full contraction patterns. The first solver call is intentionally only one timestep: it pays the Numba compilation cost before the timed production run.


In [ ]:
n_periods = 5.0
T_period = 2.0 * np.pi / float(maps.OMEGA)
T_max = np.float32(n_periods * T_period)

dt = np.float32(1.0e-4)
save_every = max(1, int(round(2.0e-2 / float(dt))))
num_steps = int(round(float(T_max) / float(dt)))

k_contact = np.float32(1.0e4)
gamma_contact = np.float32(20.0)
k_w = np.float32(1.0e4)
gamma_w = np.float32(20.0)
g = np.array([0.0, -9.81], dtype=np.float32)

r_cap = np.float32(0.03)
r_exit = np.float32(0.06)
box_padding = 0.50

def solver_box_from_time_varying_boundary(n_t=96, n_boundary=900, pad=0.50):
    period = 2.0 * np.pi / float(maps.OMEGA)
    times = np.linspace(0.0, period, int(n_t), endpoint=False)
    pts = []
    for ti in times:
        pts.append(pp.boundary_curve(t=float(ti), n=int(n_boundary)))
    pts = np.vstack(pts)
    return np.array(
        [
            pts[:, 0].min() - pad,
            pts[:, 0].max() + pad,
            pts[:, 1].min() - pad,
            pts[:, 1].max() + pad,
        ],
        dtype=np.float32,
    )

box = solver_box_from_time_varying_boundary(pad=box_padding)

print(f"period = {T_period:.6f}")
print(f"T_max  = {float(T_max):.6f}  ({n_periods:g} periods)")
print(f"dt     = {float(dt):.6g}")
print(f"steps  = {num_steps}")
print(f"saves  = {num_steps // save_every + 1}")
print("box    =", box)


## Warm-Up One Step


In [ ]:
warmup_start = time.perf_counter()
_ = csol.simulate_curvilinear_particles(
    box=box,
    q0=q0,
    p0=p0,
    m=m,
    rad=rad,
    dt=dt,
    T_max=dt,
    group_mobile=group_mobile,
    k_contact=k_contact,
    gamma_contact=gamma_contact,
    k_w=k_w,
    gamma_w=gamma_w,
    g=g,
    save_every=1,
    r_cap=r_cap,
    r_exit=r_exit,
)
warmup_runtime = time.perf_counter() - warmup_start
print(f"one-step warm-up/JIT runtime: {warmup_runtime:.3f} s")


## Run Four Periods


In [ ]:
run_start = time.perf_counter()
t_out, q_out, p_out, x_out, v_out = csol.simulate_curvilinear_particles(
    box=box,
    q0=q0,
    p0=p0,
    m=m,
    rad=rad,
    dt=dt,
    T_max=T_max,
    group_mobile=group_mobile,
    k_contact=k_contact,
    gamma_contact=gamma_contact,
    k_w=k_w,
    gamma_w=gamma_w,
    g=g,
    save_every=save_every,
    r_cap=r_cap,
    r_exit=r_exit,
)
simulation_runtime = time.perf_counter() - run_start
particle_steps = num_steps * q0.shape[0]

simulation_result = {
    "t": t_out,
    "q": q_out,
    "p": p_out,
    "x": x_out,
    "v": v_out,
    "box": box,
    "runtime": simulation_runtime,
    "dt": float(dt),
    "T_period": T_period,
    "n_periods": n_periods,
}

print(f"simulation runtime: {simulation_runtime:.3f} s")
print(f"steps per second: {num_steps / simulation_runtime:,.1f}")
print(f"particle-steps per second: {particle_steps / simulation_runtime:,.1f}")
print(f"saved frames: {len(t_out)}")
print(f"final time: {float(t_out[-1]):.6f}")


## Static Figures


In [ ]:
# ------------------------------------------------------------------
# Figure for the paper:
# two times shown in both physical and curvilinear coordinates
# ------------------------------------------------------------------

def frame_id_at_time(time_value):
    return int(np.argmin(np.abs(t_out - float(time_value))))


def draw_boundary(ax, time_value, **kwargs):
    boundary = pp.boundary_curve(t=float(time_value), n=1200)
    ax.plot(boundary[:, 0], boundary[:, 1], **kwargs)
    return boundary


def draw_q_particles(ax, frame_id, s=3):
    pts = np.column_stack(
        (
            q_out[frame_id, :, 1] % (2.0 * np.pi),
            q_out[frame_id, :, 0],
        )
    )
    ax.scatter(pts[:, 0], pts[:, 1], s=s, zorder=3)


def draw_q_grid(ax, time_value, n_x=8, n_y=5, n_points=400):
    pp.draw_curvilinear_cartesian_grid(
        ax,
        t=float(time_value),
        box=GRID_BOX,
        n_x=n_x,
        n_y=n_y,
        n_points=n_points,
        r_min=0.02,
        r_max=1.0,
        color="0.72",
        lw=0.65,
        alpha=0.75,
        zorder=0,
    )


def draw_physical_cartesian_grid(ax, box, n_x=8, n_y=5):
    x_vals = np.linspace(float(box[0]), float(box[1]), n_x)
    y_vals = np.linspace(float(box[2]), float(box[3]), n_y)

    for xv in x_vals:
        ax.plot(
            [xv, xv],
            [box[2], box[3]],
            color="0.82",
            lw=0.65,
            alpha=0.75,
            zorder=0,
        )

    for yv in y_vals:
        ax.plot(
            [box[0], box[1]],
            [yv, yv],
            color="0.82",
            lw=0.65,
            alpha=0.75,
            zorder=0,
        )


# ------------------------------------------------------------------
# Select a fixed physical Cartesian grid and two representative times.
# The grid range is based on the boundary, not on the larger simulation box.
# ------------------------------------------------------------------

GRID_BOX = pp.cartesian_grid_box_from_boundary_times(
    np.linspace(0.0, T_period, 64, endpoint=False),
    n=900,
    margin=0.04,
    mode="intersection",
)

paper_times = [
    1.3 * T_period,
    3.0 * T_period,
]

paper_frame_ids = [frame_id_at_time(tt) for tt in paper_times]

fig, axes = plt.subplots(
    2,
    2,
    figsize=(11, 7),
)

for col, frame_id in enumerate(paper_frame_ids):

    ti = float(t_out[frame_id])

    # --------------------------------------------------------------
    # Physical space
    # --------------------------------------------------------------

    ax = axes[0, col]

    draw_physical_cartesian_grid(
        ax,
        box,
        n_x=8,
        n_y=5,
    )

    draw_boundary(
        ax,
        ti,
        color="black",
        lw=1.2,
    )

    ax.scatter(
        x_out[frame_id, :, 0],
        x_out[frame_id, :, 1],
        s=3,
        zorder=3,
    )

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(float(box[0]), float(box[1]))
    ax.set_ylim(float(box[2]), float(box[3]))

    ax.set_title(f"physical space, t = {ti:.3f}")

    ax.set_xlabel("x")

    if col == 0:
        ax.set_ylabel("y")

    ax.grid(False)

    # --------------------------------------------------------------
    # Curvilinear space
    # --------------------------------------------------------------

    ax = axes[1, col]

    draw_q_grid(
        ax,
        ti,
        n_x=8,
        n_y=5,
    )

    draw_q_particles(
        ax,
        frame_id,
        s=3,
    )

    ax.axhline(
        1.0,
        color="black",
        lw=1.0,
        ls="--",
        zorder=2,
    )

    ax.set_xlim(0.0, 2.0 * np.pi)
    ax.set_ylim(0.0, 1.05)

    ax.set_title(f"curvilinear space, t = {ti:.3f}")

    ax.set_xlabel(r"$\theta$")

    if col == 0:
        ax.set_ylabel(r"$r$")

    ax.grid(False)

fig.tight_layout()

fig.savefig(
    FIGURE_DIR / "case_3_physical_curvilinear_two_times.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## Videos

The physical and curvilinear views are saved as a single side-by-side animation.


In [ ]:
from matplotlib.collections import LineCollection


def video_frame_ids(t, max_frames=360):
    n_frames = min(len(t), int(max_frames))
    ids = np.linspace(0, len(t) - 1, n_frames, dtype=int)
    return np.unique(ids)


def save_animation_file(ani, stem, fps=25, dpi=100):
    gif_path = VIDEO_DIR / f"{stem}.gif"

    writer = animation.PillowWriter(fps=fps)

    def progress(current_frame, total_frames):
        if current_frame % 25 == 0:
            print(f"saving frame {current_frame}/{total_frames}")

    ani.save(
        gif_path,
        writer=writer,
        dpi=dpi,
        progress_callback=progress,
    )

    print(f"saved {gif_path.resolve()}")
    return gif_path


def draw_physical_cartesian_grid(ax, box, n_x=8, n_y=5):
    x_vals = np.linspace(float(box[0]), float(box[1]), n_x)
    y_vals = np.linspace(float(box[2]), float(box[3]), n_y)

    for xv in x_vals:
        ax.plot(
            [xv, xv],
            [box[2], box[3]],
            color="0.82",
            lw=0.65,
            alpha=0.75,
            zorder=0,
        )

    for yv in y_vals:
        ax.plot(
            [box[0], box[1]],
            [yv, yv],
            color="0.82",
            lw=0.65,
            alpha=0.75,
            zorder=0,
        )


frame_ids = video_frame_ids(t_out, max_frames=360)

if "GRID_BOX" not in globals():
    GRID_BOX = pp.cartesian_grid_box_from_boundary_times(
        np.linspace(0.0, T_period, 64, endpoint=False),
        n=900,
        margin=0.04,
        mode="intersection",
    )

# Precompute frame-dependent geometry.
boundary_frames = []
q_grid_frames = []

for frame_id in frame_ids:
    ti = float(t_out[frame_id])

    boundary_frames.append(
        pp.boundary_curve(t=ti, n=900)
    )

    q_grid_frames.append(
        pp.cartesian_grid_curves_in_curvilinear(
            t=ti,
            box=GRID_BOX,
            n_x=8,
            n_y=5,
            n_points=500,
            r_min=0.02,
            r_max=1.0,
        )
    )

# Side-by-side video: physical space on the left, curvilinear space on the right.
fig_video, (ax_phys, ax_q) = plt.subplots(1, 2, figsize=(13, 6))

# Physical panel.
ax_phys.set_xlim(float(box[0]), float(box[1]))
ax_phys.set_ylim(float(box[2]), float(box[3]))
ax_phys.set_aspect("equal", adjustable="box")
ax_phys.set_xlabel("x")
ax_phys.set_ylabel("y")
ax_phys.set_title("physical space")
ax_phys.grid(False)

draw_physical_cartesian_grid(ax_phys, box, n_x=8, n_y=5)

boundary_line, = ax_phys.plot([], [], color="black", lw=1.2, zorder=2)
scat_phys = ax_phys.scatter([], [], s=3, zorder=3)
time_text_phys = ax_phys.text(0.02, 0.95, "", transform=ax_phys.transAxes)

# Curvilinear panel.
ax_q.set_xlim(0.0, 2.0 * np.pi)
ax_q.set_ylim(0.0, 1.05)
ax_q.set_xlabel(r"$\theta$")
ax_q.set_ylabel(r"$r$")
ax_q.set_title("curvilinear space")
ax_q.grid(False)

ax_q.axhline(1.0, color="black", lw=1.0, ls="--", zorder=2)

q_grid_collection = LineCollection(
    [],
    colors="0.72",
    linewidths=0.65,
    alpha=0.75,
    zorder=0,
)
ax_q.add_collection(q_grid_collection)

scat_q = ax_q.scatter([], [], s=3, zorder=3)
time_text_q = ax_q.text(0.02, 0.95, "", transform=ax_q.transAxes)


def init_video():
    boundary_line.set_data([], [])
    scat_phys.set_offsets(np.empty((0, 2)))

    q_grid_collection.set_segments([])
    scat_q.set_offsets(np.empty((0, 2)))

    time_text_phys.set_text("")
    time_text_q.set_text("")

    return (
        boundary_line,
        scat_phys,
        q_grid_collection,
        scat_q,
        time_text_phys,
        time_text_q,
    )


def update_video(frame):
    frame_id = frame_ids[frame]
    ti = float(t_out[frame_id])

    boundary = boundary_frames[frame]
    boundary_line.set_data(boundary[:, 0], boundary[:, 1])
    scat_phys.set_offsets(x_out[frame_id])
    time_text_phys.set_text(f"t = {ti:.3f}")

    q_grid_collection.set_segments(q_grid_frames[frame])

    pts = np.column_stack(
        (
            q_out[frame_id, :, 1] % (2.0 * np.pi),
            q_out[frame_id, :, 0],
        )
    )
    scat_q.set_offsets(pts)
    time_text_q.set_text(f"t = {ti:.3f}")

    return (
        boundary_line,
        scat_phys,
        q_grid_collection,
        scat_q,
        time_text_phys,
        time_text_q,
    )


ani_video = animation.FuncAnimation(
    fig_video,
    update_video,
    frames=len(frame_ids),
    init_func=init_video,
    interval=1000 / 25,
    blit=True,
)

combined_video_path = save_animation_file(
    ani_video,
    "case_3_physical_curvilinear_space",
    fps=25,
    dpi=100,
)

plt.close(fig_video)

combined_video_path